# Lab 2 — THE LINE
**هندسة تطبيقات النماذج اللغوية الكبيرة**  
**إعداد: أ. لطيفة سعد**

## ① تجهيز أكواد Lab 1

In [1]:
from dataclasses import dataclass

@dataclass
class LLMRequest:
    messages: list[dict]
    model: str
    max_tokens: int = 512

@dataclass
class LLMResponse:
    text: str

class OpenAICompatClient:
    def __init__(self, name="OpenAI"):
        self.name = name

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        question = request.messages[-1]["content"]
        return LLMResponse(
            text=f"{self.name} response to: {question}"
        )

In [2]:
the_line = OpenAICompatClient()

request = LLMRequest(
    messages=[
        {
            "role": "user",
            "content": "كيف يمكن تقليل استهلاك الطاقة في مبنى مستدام قريب من تبوك؟"
        }
    ],
    model="demo-model"
)

response = the_line.complete(request)
print(response.text)

OpenAI response to: كيف يمكن تقليل استهلاك الطاقة في مبنى مستدام قريب من تبوك؟


## ② AnthropicClient — الـAdapter الثاني

In [3]:
class AnthropicClient:
    def __init__(self, name="Anthropic"):
        self.name = name

    def build_payload(self, request: LLMRequest):
        system = ""
        turns = []
        for message in request.messages:
            if message["role"] == "system":
                system = message["content"]
            else:
                turns.append(message)
        return {
            "model": request.model,
            "system": system,
            "max_tokens": request.max_tokens,
            "messages": turns
        }

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        payload = self.build_payload(request)
        question = payload["messages"][-1]["content"]
        return LLMResponse(
            text=f"{self.name} response to: {question}"
        )

In [4]:
request = LLMRequest(
    messages=[
        {
            "role": "system",
            "content": "You are THE LINE, a sustainability engineering assistant for energy, water, materials, and smart infrastructure."
        },
        {
            "role": "user",
            "content": "كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟"
        }
    ],
    model="demo-model"
)

anthropic = AnthropicClient()
print(anthropic.build_payload(request))

{'model': 'demo-model', 'system': 'You are THE LINE, a sustainability engineering assistant for energy, water, materials, and smart infrastructure.', 'max_tokens': 512, 'messages': [{'role': 'user', 'content': 'كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟'}]}


## ③ نفس الطلب على المزوّدين

In [5]:
clients = [
    OpenAICompatClient(),
    AnthropicClient()
]

for client in clients:
    response = client.complete(request)
    print(response.text)

OpenAI response to: كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟
Anthropic response to: كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟


## ④ إضافة vLLM — النموذج المفتوح

In [6]:
routes = {
    "primary": AnthropicClient(),
    "cheap": OpenAICompatClient(),
    "vllm": OpenAICompatClient(name="vLLM")
}

for route, client in routes.items():
    response = client.complete(request)
    print(route, "→", response.text)

primary → Anthropic response to: كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟
cheap → OpenAI response to: كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟
vllm → vLLM response to: كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟


## ⑤ الـStreaming وقياس TTFT

In [7]:
import time

def stream_chat(client, request):
    response = client.complete(request)
    words = response.text.split()

    start = time.time()
    ttft = None
    time.sleep(0.4)

    for word in words:
        if ttft is None:
            ttft = time.time() - start
        print(word, end=" ", flush=True)
        time.sleep(0.3)

    total = time.time() - start
    print()
    print("TTFT :", round(ttft, 1), "seconds")
    print("Total:", round(total, 1), "seconds")

stream_chat(routes["primary"], request)

Anthropic 

response 

to: 

كيف 

يمكن 

...

على 

خفض 

استهلاك 

الطاقة؟ 


TTFT : 0.4 seconds
Total: 4.9 seconds


## ⑥ الـRetry والـFallback

In [8]:
class BrokenClient:
    def __init__(self, name="Anthropic"):
        self.name = name

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        raise Exception("429 Rate limit")

class ResilientClient:
    def __init__(self, primary, fallback, max_attempts=3):
        self.primary = primary
        self.fallback = fallback
        self.max_attempts = max_attempts

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        for attempt in range(1, self.max_attempts + 1):
            try:
                return self.primary.complete(request)
            except Exception as error:
                print("Retry", attempt, "→", error)
                time.sleep(1)

        print("Fallback → vllm")
        return self.fallback.complete(request)

In [9]:
the_line = ResilientClient(
    primary=BrokenClient(),
    fallback=routes["vllm"]
)

response = the_line.complete(request)
print(response.text)

Retry 1 → 429 Rate limit


Retry 2 → 429 Rate limit


Retry 3 → 429 Rate limit


Fallback → vllm
vLLM response to: كيف يمكن اختيار مواد مناسبة للبيئة المحلية وتساعد على خفض استهلاك الطاقة؟


## ⑦ الـBenchmark

In [10]:
delays = {
    "primary": 1.0,
    "cheap": 0.6,
    "vllm": 1.4
}

questions = [
    "كيف يمكن تقليل استهلاك الطاقة في مبنى مستدام قريب من تبوك؟",
    "How can treated water be reused in a sustainable project?",
    "ما المواد التي تساعد على تقليل الحمل الحراري في المباني؟"
]

for route, client in routes.items():
    start = time.time()

    for question in questions:
        request = LLMRequest(
            messages=[
                {
                    "role": "user",
                    "content": question
                }
            ],
            model="demo-model"
        )
        time.sleep(delays[route])
        client.complete(request)

    average = (time.time() - start) / len(questions)
    print(route, "| average:", round(average, 1), "seconds")

primary | average: 1.0 seconds


cheap | average: 0.6 seconds


vllm | average: 1.4 seconds
